# SemanticKV-Reorg · Kaggle 一键上机脚本（V1.5）

**用途**：在 Kaggle（免费 T4×2 / P100，或租用的 A100/4090）上验证 SemanticKV-Reorg 的 V1.5 语义压缩机制。

## 三步用法
1. 左侧 **+ Add Input** → **Upload** → 上传 `semantickv-reorg.tar.gz`（或作为 Dataset 添加）；
2. 右上 **Settings** → Accelerator 选 **GPU T4 x2**（或 P100）；
3. **Run All**。

## baseline-first 两步流程
1. **Step 1 · 原始 needle 基线**（`--baseline-only`）：先确认无压缩模型在该上下文长度下能检索 needle。
   原始 MISS ⇒ 测试无区分度，请换更大模型（Settings 改模型名）或缩短上下文后重跑；
2. **Step 2 · V1.5 压缩验证**：通过基线门后才跑压缩全流程（V0 不变性 + 分组扫描 + 保真度 + 端到端 needle）。

## V1.5 策略要点（相对 v0.1.2 的四处修正）
- **度量**：cosine → **centered**（减全局均值再归一化，修复 LLM 隐空间各向异性）；
- **代表**：flat 群均值 → **Top-K 代表 token**（每群注意力入度最高的 K 个真实 KV，K=8 推荐，K=4 更省）；
- **结构**：Attention Sink 保护（前 4 token 不分组）+ 近期滑窗（64 token 不压缩）；
- **层级**：只压中层（24 层模型压 6-17，即 25%-75%），保留底层/顶层原样；
- **分数先验**：群均值槽补 `log(群规模)` 分数修正（少槽 softmax 温度剧变的补偿）。

> 边界声明：本脚本验证**机制与数学正确性**，不是最终性能；架构有效性与模型参数量无关。


In [ ]:
# ============ 1. 安装依赖 ============
!pip install -q -U transformers scipy pyyaml numpy

import os, sys, glob
# 国内网络如无法访问 huggingface.co，取消下一行注释：
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
print("环境准备完成")


In [ ]:
# ============ 2. 定位仓库 ============
# 方式 A：semantickv-reorg.tar.gz 上传到 /kaggle/working
# 方式 B：tar.gz 作为 Dataset 挂到 /kaggle/input/<数据集名>/
cands = glob.glob("/kaggle/working/semantickv-reorg.tar.gz") + \
        glob.glob("/kaggle/input/*/semantickv-reorg.tar.gz")
if cands:
    !tar -xzf {cands[0]} -C /kaggle/working/
    REPO = "/kaggle/working/semantickv-reorg"
else:
    # 方式 C：已推送远端仓库时，改成真实地址
    !git clone --depth 1 https://github.com/<your>/semantickv-reorg.git /kaggle/working/semantickv-reorg
    REPO = "/kaggle/working/semantickv-reorg"
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "src", "semantickv"))
assert os.path.exists(os.path.join(REPO, "eval_runner.py")), "仓库解压失败"
print("仓库路径:", REPO)


In [ ]:
# ============ 3. GPU 检查 + 接口签名校验 + 选模型 ============
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name}  显存 {vram_gb:.1f} GB")
else:
    vram_gb = 0.0
    print("未检测到 GPU，将使用 CPU 慢速模式（建议选小模型 + 短上下文）")

# 按显存自动选模型（7B 起步：0.5B 的 needle 基线不可靠，见 results/REAL_MODEL_VALIDATION.md）
if vram_gb >= 40:
    MODEL, DTYPE, CTX = "Qwen/Qwen2.5-7B-Instruct", "bfloat16", 4096
elif vram_gb >= 14:
    MODEL, DTYPE, CTX = "Qwen/Qwen2.5-1.5B-Instruct", "bfloat16", 2048
else:
    MODEL, DTYPE, CTX = "Qwen/Qwen2.5-0.5B", "float32", 1024
print(f"自动选择: {MODEL} dtype={DTYPE} ctx={CTX}")
DEV = 'cuda' if vram_gb > 0 else 'cpu'

# 接口签名校验（13/13 通过才能继续）
!cd {REPO} && python3 eval_runner.py --check-interfaces


In [ ]:
# ============ 4. Step 1：原始 needle 基线（baseline 门） ============
# 无压缩、无分组、无压缩机制，只验证原始模型在该 ctx 下能否检索 needle。
# 耗时参考：0.5B/CPU 约 1 分钟；7B/A100 约 1 分钟。
!cd {REPO} && SEMANTICKV_THREADS=8 python3 src/semantickv/v41_torch_native_runner.py \
    --model {MODEL} --dtype {DTYPE} --device {DEV} \
    --ctx {CTX} --baseline-only \
    --needle-positions 0.15,0.5,0.85 \
    --out results/kaggle_baseline.json

import json
bl = json.load(open(os.path.join(REPO, "results", "kaggle_baseline.json")))
base_hits = [r["original_hit"] for r in bl["stages"]["needle"]]
print("\n===== Step 1 结果：原始 needle 基线 =====")
for r in bl["stages"]["needle"]:
    print(f"pos={r['position_frac']}: {'HIT ' if r['original_hit'] else 'MISS'} {r['original_answer']!r}")
# DS-20261018-011 前置门禁：7B 无压缩基线 needle 命中 ≥ 2/3 方有验证资格
n_base_hit = sum(1 for h in base_hits if h)
if n_base_hit >= 2:
    print(f"\n>>> 前置门禁通过（≥2/3 口径）：{n_base_hit}/3 HIT，继续 Step 2/3。")
else:
    print(f"\n>>> 前置门禁未通过（需 ≥2/3，实得 {n_base_hit}/3）。")
    print(">>> 原始模型 needle 检索能力不足，请换更大模型（Settings 改模型名）")
    print(">>> 或缩短 ctx 后重新 Run All。")


In [ ]:
# ============ 5. Step 2：V1.5 全量验证 ============
# V0 重排不变性 + (k,τ) 扫描 + V1/V1.5 保真度曲线 + 端到端 needle + 真实显存双指标
# 耗时参考：0.5B/CPU 约 3-4 分钟；1.5B/T4 约 2 分钟；7B/A100 约 3-4 分钟
!cd {REPO} && SEMANTICKV_THREADS=8 python3 src/semantickv/v41_torch_native_runner.py \
    --model {MODEL} --dtype {DTYPE} --device {DEV} \
    --ctx {CTX} --strategy v15 \
    --metric centered --compress-mode topk --topk-per-group 8 \
    --sink-tokens 4 --window-tokens 64 --compress-layers 0.25-0.75 \
    --kmean-renorm none --group-score-prior logcount \
    --k-grid 8 --tau-grid 0.97 \
    --needle-positions 0.15,0.5,0.85 \
    --out results/real_model_v15.json


In [ ]:
# ============ 6. Step 3：Router 空壳 + 插件冒烟（H-DS15 / H-DS17 · DS-20261018-011） ============
# 挂随机初始化的 DS 规格 Router 空壳（learned_router.SemanticRouter，纯 Linear+ReLU），
# 验证四件事（DS-011 验收标准）：
#   ① Router 单次前向延迟（通过线 <0.5ms；证伪线 >2ms）
#   ② 参数量 <50M
#   ③ 权重占 7B KV cache 比例 <10%
#   ④ CUDA Graph capture 不被破坏（算子级 smoke；插件级见 vllm 分支）
# 存证：results/kaggle_7b_vllm_plugin_bench.json
import json, time
import torch
import torch.nn as nn
from src.semantickv.vllm_plugin.learned_router import LearnedSemanticRouter

Q_DIM = {"Qwen/Qwen2.5-7B-Instruct": 3584, "Qwen/Qwen2.5-1.5B-Instruct": 1536,
         "Qwen/Qwen2.5-0.5B": 896}.get(MODEL, 1024)
router = LearnedSemanticRouter(q_dim=Q_DIM, hidden_dim=256, num_groups=64).to(DEV).eval()
n_params = router.num_parameters()

@torch.no_grad()
def _lat(n_tok, reps=200):
    x = torch.randn(1, n_tok, Q_DIM, device=DEV)
    for _ in range(20):
        router(x)
    if DEV == "cuda":
        torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        router(x)
        if DEV == "cuda":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000.0)
    ts.sort()
    return {"p50_ms": round(ts[len(ts)//2], 4),
            "p99_ms": round(ts[min(len(ts)-1, int(len(ts)*0.99))], 4)}

lat_n1, lat_n2048 = _lat(1), _lat(2048)
weight_mb = n_params * router.net[0].weight.element_size() / 1024**2
kv_mb = 28 * 2 * 8 * 128 * 2 * 4096 / 1024**2   # 7B/28层/8KV头/hd128/fp16 KV/ctx4096

# ---- ④ CUDA Graph capture 探测（算子级 smoke） ----
cuda_graph = {"status": "skipped", "reason": "cpu 环境"}
if DEV == "cuda":
    try:
        g = torch.cuda.CUDAGraph()
        x = torch.randn(1, 1, Q_DIM, device=DEV)
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                router(x)
        torch.cuda.current_stream().wait_stream(s)
        with torch.cuda.graph(g):
            y_cap = router(x)
        g.replay()
        torch.cuda.synchronize()
        same = bool(torch.allclose(y_cap, router(x), atol=1e-5))
        cuda_graph = {"status": "captured" if same else "mismatch",
                      "replay_matches_eager": same}
    except Exception as e:               # noqa: BLE001
        cuda_graph = {"status": "failed", "error": str(e)}

# ---- 插件级（vLLM）探测：可用则测 enforce_eager=False，不可用记 pending ----
vllm_status = "pending（Kaggle 上 pip install vllm 后按 vllm_plugin/README 挂载点实测）"
try:
    import vllm
    vllm_status = f"vllm {vllm.__version__} 可用——按 README 挂载 SemanticRouter 后重跑 decode bench"
except Exception:                        # noqa: BLE001
    pass

bench = {
    "config": {"device": DEV, "dtype": str(next(router.parameters()).dtype),
               "model": MODEL, "q_dim": Q_DIM, "ctx": CTX,
               "purpose": "H-DS15/H-DS17 Router 空壳 + 插件冒烟（DS-20261018-011-WB1）"},
    "environment": {"kind": "real-model-arch-smoke",
                    "note": "Router 为随机初始化空壳；主模型未加载（算子级冒烟）"},
    "router": {"arch": "ds011_linear_relu_linear", "params": n_params,
               "params_under_50m": n_params < 50_000_000,
               "latency_single_query_N1": lat_n1,
               "latency_prefill_N2048": lat_n2048,
               "pass_line_0p5ms": lat_n1["p99_ms"] < 0.5,
               "fail_line_2ms": lat_n1["p99_ms"] > 2.0,
               "weight_mb": round(weight_mb, 4),
               "kv_cache_mb_7b_ctx4096_fp16": round(kv_mb, 2),
               "kv_mem_pct": round(100.0 * weight_mb / kv_mb, 6),
               "kv_mem_fail_10pct": 100.0 * weight_mb / kv_mb > 10.0},
    "cuda_graph": cuda_graph,
    "vllm_plugin": vllm_status,
    "summary": {
        "params_under_50m": n_params < 50_000_000,
        "kv_mem_under_10pct": 100.0 * weight_mb / kv_mb <= 10.0,
        "latency_pass_0p5ms": lat_n1["p99_ms"] < 0.5,
        "cuda_graph_ok": cuda_graph.get("status") == "captured",
    },
}
dest = os.path.join(REPO, "results", "kaggle_7b_vllm_plugin_bench.json")
os.makedirs(os.path.dirname(dest), exist_ok=True)
with open(dest, "w", encoding="utf-8") as f:
    json.dump(bench, f, ensure_ascii=False, indent=2)

print("===== Step 3：Router 空壳 + 插件冒烟 =====")
print(f"Router: q_dim={Q_DIM} params={n_params:,} (<50M: {bench['summary']['params_under_50m']})")
print(f"单 query 前向: p50={lat_n1['p50_ms']}ms p99={lat_n1['p99_ms']}ms "
      f"(通过线 0.5ms: {bench['summary']['latency_pass_0p5ms']})")
print(f"整段 N=2048 前向: p50={lat_n2048['p50_ms']}ms")
print(f"权重 {weight_mb:.3f}MB vs 7B KV cache {kv_mb:.0f}MB -> "
      f"{bench['router']['kv_mem_pct']}% (证伪线 10%)")
print(f"CUDA Graph: {cuda_graph}")
print(f"vLLM 插件: {vllm_status}")


In [ ]:
# ============ 7. 结果摘要（V1 vs V1.5 对照 + 显存双指标） ============
d = json.load(open(os.path.join(REPO, "results", "real_model_v15.json")))
bl = json.load(open(os.path.join(REPO, "results", "kaggle_baseline.json")))
print(f"模型: {d['model']}   总耗时: {d['wall_time_s']}s   峰值RSS: {d.get('prefill_peak_rss_mb')}MB")
cm = d.get("cuda_memory", {})
if cm:
    print(f"CUDA 峰值显存: allocated={cm.get('max_memory_allocated_mb')}MB "
          f"reserved={cm.get('max_memory_reserved_mb')}MB（reserved>allocated 属缓存池正常现象）")

f = d["stages"]["fidelity"]
v0 = f["v0_reorder_invariance"]
print(f"\n[V0 重排不变性] cos mean={v0['mean']:.6f} min={v0['min']:.6f}  "
      f"{'PASS 机制正确' if v0['mean'] > 0.999 else 'WARN 低于预期'}")

print("\n[压缩-保真度曲线]（单层注意力输出 vs 原始，target layer=12）")
print(f"{'k':>3} {'tau':>5} {'G':>5} {'V1压缩':>7} {'V1 cos':>8} | "
      f"{'flat+sink':>9} {'weighted':>9} {'topk':>8} {'topk压缩':>8}")
for r in f["sweep"]:
    v15 = r["v15"]
    print(f"{r['k']:>3} {r['tau']:>5.2f} {r['n_groups']:>5} {r['compression_ratio']:>6.0f}x "
          f"{r['v1_group_mean']['mean']:>8.4f} | "
          f"{v15['flat_mean_sink']['mean']:>9.4f} {v15['weighted_sink']['mean']:>9.4f} "
          f"{v15['topk_sink']['mean']:>8.4f} {v15['topk_compression']:>7.1f}x")

print("\n[needle 检索] 基线 vs 语义压缩（Step 1 门 + Step 2 端到端）")
base_by_pos = {r["position_frac"]: r for r in bl["stages"]["needle"]}
for r in d["stages"]["needle"]:
    pf = r.get("prefill_fidelity", {})
    b = base_by_pos.get(r["position_frac"], {})
    kv = r.get("kv_bytes", {})
    print(f"pos={r['position_frac']}: 原始 {'HIT' if r['original_hit'] else 'MISS'} {r['original_answer']!r} | "
          f"压缩 {'HIT' if r['compressed_hit'] else 'MISS'} {r['compressed_answer']!r}")
    print(f"   hidden cos={pf.get('final_hidden_cos', {}).get('mean', float('nan')):.4f} "
          f"argmax一致率={pf.get('logits_argmax_agreement')} | "
          f"KV {kv.get('original_mb')}MB->{kv.get('compressed_mb')}MB 省{kv.get('saving_pct')}%")

print("\n[GSC (k,tau) 扫描]（真实隐状态分组质量，E0 校准输入）")
for r in d["stages"]["gsc_scan"]["rows"]:
    print(f"k={r['k']:>2} tau={r['tau']:.2f}: G={r['n_groups']:>4} "
          f"ARI_topic={r['ari_topic']:+.4f} n_groups_per_tok={r.get('groups_per_tok')}")

n_hit = sum(1 for r in d["stages"]["needle"] if r["compressed_hit"])
n_all = len(d["stages"]["needle"])
print(f"\n===== 总结论：压缩 needle {n_hit}/{n_all} HIT；"
      f"保真度 topk={f['sweep'][0]['v15']['topk_sink']['mean']:.4f}（V1 基线 {f['sweep'][0]['v1_group_mean']['mean']:.4f}）=====")


## 结果解读指南

| 指标 | 期望 | 含义 |
|---|---|---|
| Step 1 基线 | 原始 HIT | 不过则测试无区分度，换大模型（7B 起步） |
| V0 cos | ≈1.000000 | KV gather/scatter + RoPE + GQA + 掩码的**张量机制**正确（不过则代码有 bug） |
| V1 cos | 基线参照 | flat 群均值、cosine 度量、全层压缩（v0.1.2 方案） |
| topk cos | > V1 | V1.5：Top-K 真实代表 + centered 度量 + sink/窗口保护 + 中层压缩 |
| hidden cos / argmax | 越高越好 | 端到端保真度：最终隐状态余弦 / logits argmax 一致率 |
| needle | 基线 HIT 且压缩 HIT | 理想情况；压缩 HIT 率随 KV 节省率 trade-off |
| KV saving | 30%-50% 现实区间 | 12/24 层压缩 + Top-K 后的真实 KV 字节节省（>60% 不可信，检查记账） |
| ARI_topic | >0.5 好 | GSC 在真实隐状态上恢复话题结构的程度（合成数据是 1.0） |

**V1 vs V1.5 对照逻辑**（v0.1.2 → v0.2.0 的四处修正，详见 `results/REAL_MODEL_VALIDATION_V15.md`）：
1. 若 `topk cos > V1 cos`：Top-K 真实代表 + centered 度量优于 flat 群均值 + cosine（预期，0.5B 实测 0.61 vs 0.41）；
2. 若 `topk cos` 仍 <0.7：检查 sink/窗口是否真的保护了特权 token（`n_priv` 字段）；
3. 若 needle 仍 MISS 但 hidden cos >0.8：问题不在压缩保真度，在检索鲁棒性——尝试加大 `--topk-per-group` 到 12/16，或加宽 `--window-tokens` 到 128；
4. 真实显存看 `reserved`（分配器缓存池上限）而非 `allocated`（张量实际占用），两者差 30% 属正常。

**已知边界**：
- 0.5B 模型 needle 检索能力弱（v0.1.2 实测原始即部分 MISS），正式结论以 7B 为准；
- 慢速验证器为保持 RoPE 位置正确仍写全量 cache，KV 节省以解析记账（`kv_bytes`）为准；
- 真实隐空间呈「话题 × 位置」多尺度结构，单尺度 GSC 的 ARI 明显低于合成数据——这正是 E0 要做 kNN 分布校准的原因。
